# Milestone 2: fixed-depth relational baseline

Import this notebook from GitHub into Kaggle. Enable **GPU** and **internet access** before running it. It uses only `cuda:0`, including on a two-GPU runtime, and preserves the installed PyTorch stack.

Push the notebook and package changes to GitHub first. Run cells in order, or use **Run All** after reviewing the settings. No cell evaluates the research test split. Notebook outputs are intentionally empty in Git.

Run `kaggle_relational_smoke.ipynb` successfully first. This notebook implements the [recorded protocol](https://github.com/Krailon/multi-modal-loop-llm/blob/milestone2/docs/milestones/milestone2.md): **10 total epochs, R=2, 2,880 updates**. It evaluates the final checkpoint on validation with seeds 0–4 and displays the agreed gates. It never extends the budget to improve results or selects an earlier checkpoint.

A partial local checkpoint resumes automatically; a completed one skips training. Existing reports are reused only after provenance checks. A failed gate remains a valid baseline result to diagnose. Frozen-test evaluation is a separate later step.


## 1. Settings

Keep the experiment settings in the recorded protocol unchanged. Paths and the code reference are the intended notebook settings.


In [ ]:
# Choose a pushed branch/tag or pin an exact commit for reproducibility.
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"
REPO_DIR = "/kaggle/working/multi-modal-loop-llm"
RUN_ROOT = "/kaggle/working/milestone2_baseline"
# Optional checkpoint from an earlier session (for example under /kaggle/input/...).
# When set, RUN_ROOT must be fresh. Otherwise RUN_ROOT/training/last.pt resumes automatically.
RESUME_CHECKPOINT = None

## 2. Checkout and package setup

An existing checkout must be clean and match the requested revision. If the branch moved since your run, use the recorded commit as `REPO_REF` to resume. Setup never deletes a checkout or switches an existing checkout to another revision.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## 3. CUDA and provenance

This checks CUDA and records the resolved revision, model/protocol settings, and runtime. It fails rather than falling back to CPU.


In [ ]:
# Detect a package cached from another checkout in this kernel.
import multimodal_loop.train.kaggle as notebook_helpers
from multimodal_loop.train.kaggle import (
    archive_run,
    evaluate_run,
    prepare_run,
    train_run,
)

if not Path(notebook_helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel before proceeding")

run = prepare_run(REPO_DIR, RUN_ROOT, resume_checkpoint=RESUME_CHECKPOINT)

## 4. Training

Commands stream into the cell and persistent log files. An error stops the notebook.


In [ ]:
checkpoint = train_run(run)
print("Checkpoint:", checkpoint)

## 5. Validation controls

Images and questions are intervened on separately. Original targets and sequence layout stay fixed. A perfect model has an expected 37.5% accuracy under ordinary within-image question shuffling, not 25%.


In [ ]:
report = evaluate_run(run)

The gate summary uses unrounded metrics: ≥90% overall accuracy, ≥80% for every question type, ≥80% all-four accuracy, and ≥30 percentage points over blank images, mean shuffled images, and mean shuffled questions. All six gates must pass. Different-answer-pair accuracy, invalid predictions, losses, and individual shuffles remain available in the full report.


## 6. Retain the artifacts

The archive includes the checkpoint, manifest, settings, metrics, validation reports, provenance, and subprocess logs. Download it using the link or find it in the notebook output files. Keep these outputs outside the temporary session before it ends. The original run directory is retained as well.


In [ ]:
from IPython.display import FileLink, display

archive = archive_run(run)
print("Artifact archive:", archive)
display(FileLink(str(archive)))

## Continuing in another session

Attach the saved artifacts, set `RESUME_CHECKPOINT` to the restored `training/last.pt`, and choose a fresh `RUN_ROOT`. Use the original recorded code commit as `REPO_REF` and the same CUDA backend. The notebook validates settings and computes the remaining part of the ten-epoch budget automatically. At epoch ten it copies the completed checkpoint into the new output and performs validation only. Preserve the original archive and logs too; a checkpoint alone cannot reconstruct the original source revision or runtime history.

To rerun this notebook in its existing output directory, leave `RESUME_CHECKPOINT = None`. Hardware/software changes should be recorded; CPU deterministic-resume tests do not promise bitwise replay across different CUDA environments.
